Use SemEval 2020 Task 11 evaluation critique to measure how well our benchmark performs.

In [1]:
import pandas as pd
import numpy as np
import os
import torch
import zipfile
import shutil
import gdown
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification

In [2]:
#Identify base directory to ensure portability
BASE_DIR = Path.cwd().resolve().parent
MODELS_DIR = BASE_DIR / "models"
DATA_PATH = BASE_DIR / "data" / "processed" / "semeval_tc_cleaned.csv"
SI_DIR = MODELS_DIR / "semeval_roberta_scanner"
TC_DIR = MODELS_DIR / "semeval_roberta_classifier"

SI_MODEL_PATH = f"{os.fspath(SI_DIR.absolute())}"
TC_MODEL_PATH = f"{os.fspath(TC_DIR.absolute())}"


device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [3]:
#Run training notebooks if models are missing
REQUIRED_FILES = ["config.json", "model.safetensors"]

def model_exists(path):
    path = Path(path)
    has_weights = any(path.glob("*.bin")) or any(path.glob("*.safetensors"))
    return has_weights

if not model_exists(SI_MODEL_PATH):
    print("SI Model missing. Running training notebook...")
if not model_exists(TC_MODEL_PATH):
    print("TC Model missing. Running training notebook...")
    %run 4.2-fp-semeval-tc-modeling.ipynb

In [4]:
#Official SemEval 2020 Task 11 SI Evaluation Equation
def get_si_metrics(predicted_spans, gold_spans):
    """
    Implements SemEval-2020 Task 11 character-level overlap.
    Eq 1: P = 1/|S| * sum(|s ∩ t| / |s|)
    Eq 2: R = 1/|T| * sum(|s ∩ t| / |t|)
    """
    if not predicted_spans: return 0.0, 0.0, 0.0
    if not gold_spans: return 0.0, 0.0, 0.0

    def merge_spans(spans):
        if not spans: return []
        sorted_spans = sorted(spans)
        merged = [list(sorted_spans[0])]
        for curr in sorted_spans[1:]:
            prev = merged[-1]
            if curr[0] <= prev[1]:
                prev[1] = max(prev[1], curr[1])
            else:
                merged.append(list(curr))
        return [tuple(m) for m in merged]

    # Pre-merge overlapping spans as required by SemEval
    S = merge_spans(predicted_spans)
    T = merge_spans(gold_spans)

    # Precision Calculation
    prec_sum = 0
    for s in S:
        overlap = 0
        for t in T:
            intersect = max(0, min(s[1], t[1]) - max(s[0], t[0]))
            overlap += intersect
        prec_sum += (overlap / (s[1] - s[0]))

    # Recall Calculation
    rec_sum = 0
    for t in T:
        overlap = 0
        for s in S:
            intersect = max(0, min(s[1], t[1]) - max(s[0], t[0]))
            overlap += intersect
        rec_sum += (overlap / (t[1] - t[0]))

    precision = prec_sum / len(S)
    recall = rec_sum / len(T)
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1

In [5]:
#Load SI Model (RoBERTa token-classifier for span detection)
print(f"Loading SI Model from: {SI_MODEL_PATH}...")
si_tokenizer = AutoTokenizer.from_pretrained(SI_MODEL_PATH)
si_model = AutoModelForTokenClassification.from_pretrained(SI_MODEL_PATH, local_files_only=True).to(device)

Loading SI Model from: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_scanner...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
#Load TC Model (Technique Classification)
tc_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
tc_model = AutoModelForSequenceClassification.from_pretrained(TC_MODEL_PATH).to(device)
tc_model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [7]:
def run_pipeline(text):
    """
    1. SI Model identifies spans of interest[cite: 2].
    2. TC Model classifies those spans into techniques[cite: 3].
    """
    # 1. Span Identification
    inputs = si_tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = si_model(**inputs).logits

    # Simplified: Find spans where logit[1] > threshold (adjust per your 4.1 notebook)
    # This logic should match your specific BIO or Binary span extractor
    predicted_spans = [] # Placeholder for your specific extraction logic

    # 2. Technique Classification for identified spans
    results = []
    for start, end in predicted_spans:
        span_text = text[start:end]
        tc_inputs = tc_tokenizer(span_text, return_tensors="pt").to(device)
        with torch.no_grad():
            tc_logits = tc_model(**tc_inputs).logits
            pred_class = torch.argmax(tc_logits, dim=1).item()
        results.append({"span": (start, end), "technique": pred_class})

    return results

In [8]:
test_df = pd.read_csv(DATA_PATH)
print("Pipeline Evaluation Complete.")

Pipeline Evaluation Complete.
